In [0]:
%sql
-- Rename the External Silver table to the original table name

ALTER TABLE adbdevbankproject.silver.cards_data_external
RENAME TO adbdevbankproject.silver.cards_data;

In [0]:
%sql
-- Drop the original Managed Silver table after validating the External table

DROP TABLE adbdevbankproject.silver.cards_data;

In [0]:
%sql
-- Check the schema of the external Silver table

DESCRIBE adbdevbankproject.silver.cards_data_external;

col_name,data_type,comment
id,int,null
client_id,int,null
card_brand,string,null
card_type,string,null
expires,date,null
has_chip,boolean,null
num_cards_issued,int,null
credit_limit,"decimal(12,2)",null
acct_open_date,date,null
year_pin_last_changed,int,null


In [0]:
%sql
-- Compare the schema of the original and external Silver tables

DESCRIBE adbdevbankproject.silver.cards_data;

col_name,data_type,comment
id,int,null
client_id,int,null
card_brand,string,null
card_type,string,null
expires,date,null
has_chip,boolean,null
num_cards_issued,int,null
credit_limit,"decimal(12,2)",null
acct_open_date,date,null
year_pin_last_changed,int,null


In [0]:
%sql
-- Compare row counts between the original Managed Silver table and the new External Silver table

SELECT 
    'Original Silver Table' AS table_name,
    COUNT(*) AS row_count
FROM adbdevbankproject.silver.cards_data

UNION ALL

SELECT 
    'External Silver Table' AS table_name,
    COUNT(*) AS row_count
FROM adbdevbankproject.silver.cards_data_external;

table_name,row_count
Original Silver Table,6146
External Silver Table,6146


In [0]:
%sql
DESCRIBE DETAIL adbdevbankproject.silver.cards_data_external;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,0aaff2cb-cb4b-458e-a20b-6c35543b4647,adbdevbankproject.silver.cards_data_external,null,abfss://silver@stgdevbankproject.dfs.core.windows.net/cards_data,2026-08-21T13:29:16.192Z,2026-08-21T13:29:17.000Z,List(),List(),1,96515,Map(delta.enableDeletionVectors -> true),3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
-- Create the external Cards Silver table in ADLS

CREATE TABLE adbdevbankproject.silver.cards_data_external
USING DELTA
LOCATION 'abfss://silver@stgdevbankproject.dfs.core.windows.net/cards_data'
AS
SELECT *
FROM adbdevbankproject.silver.cards_data;

num_affected_rows,num_inserted_rows


In [0]:
%sql
-- Check the current table type and storage location

DESCRIBE DETAIL adbdevbankproject.silver.cards_data;

format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,49b9986c-6d5e-4424-a3e0-90a61e7c67a6,adbdevbankproject.silver.cards_data,null,,2026-08-21T13:20:51.722Z,2026-08-21T13:20:54.000Z,List(),List(),1,68551,"Map(delta.parquet.format.version -> 2.12.0, delta.parquet.format.version.afe.internal -> 2.12.0, delta.parquet.compression.codec -> zstd, delta.enableDeletionVectors -> true)",3,7,"List(appendOnly, deletionVectors, invariants)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


In [0]:
%sql
DESCRIBE adbdevbankproject.silver.cards_data;

col_name,data_type,comment
id,int,null
client_id,int,null
card_brand,string,null
card_type,string,null
expires,date,null
has_chip,boolean,null
num_cards_issued,int,null
credit_limit,"decimal(12,2)",null
acct_open_date,date,null
year_pin_last_changed,int,null


In [0]:
%sql
SELECT *
FROM adbdevbankproject.silver.cards_data
LIMIT 10;

id,client_id,card_brand,card_type,expires,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,2022-12-01,true,2,24295.00,2002-09-01,2008,false
2731,825,Visa,Debit,2020-12-01,true,2,21968.00,2014-04-01,2014,false
3701,825,Visa,Debit,2024-02-01,true,2,46414.00,2003-07-01,2004,false
42,825,Visa,Credit,2024-08-01,false,1,12400.00,2003-01-01,2012,false
4659,825,Mastercard,Debit (Prepaid),2009-03-01,true,1,28.00,2008-09-01,2009,false
4537,1746,Visa,Credit,2003-09-01,true,1,27500.00,2003-09-01,2012,false
1278,1746,Visa,Debit,2022-07-01,true,2,28508.00,2011-02-01,2011,false
3687,1746,Mastercard,Debit,2022-06-01,true,2,9022.00,2003-07-01,2015,false
3465,1746,Mastercard,Debit (Prepaid),2020-11-01,true,2,54.00,2010-06-01,2015,false
3754,1746,Mastercard,Debit (Prepaid),2023-02-01,true,1,99.00,2006-07-01,2012,false


In [0]:
%sql
-- Create the cleaned Cards Silver table

CREATE OR REPLACE TABLE adbdevbankproject.silver.cards_data
AS
SELECT
    id,
    client_id,
    card_brand,
    card_type,
    TO_DATE(expires, 'MM/yyyy') AS expires,
    UPPER(has_chip) = 'YES' AS has_chip,
    CAST(num_cards_issued AS INT) AS num_cards_issued,
    CAST(REPLACE(credit_limit, '$', '') AS DECIMAL(12,2)) AS credit_limit,
    TO_DATE(acct_open_date, 'MM/yyyy') AS acct_open_date,
    CAST(year_pin_last_changed AS INT) AS year_pin_last_changed,
    LOWER(card_on_dark_web) = 'yes' AS card_on_dark_web
FROM adbdevbankproject.bronze.cards_data;

num_affected_rows,num_inserted_rows


In [0]:
%sql
DESCRIBE adbdevbankproject.bronze.cards_data;

col_name,data_type,comment
id,int,null
client_id,int,null
card_brand,string,null
card_type,string,null
card_number,bigint,null
expires,string,null
cvv,int,null
has_chip,string,null
num_cards_issued,int,null
credit_limit,string,null


In [0]:
%sql
SELECT *
FROM adbdevbankproject.bronze.cards_data
LIMIT 10;

id,client_id,card_brand,card_type,card_number,expires,cvv,has_chip,num_cards_issued,credit_limit,acct_open_date,year_pin_last_changed,card_on_dark_web
4524,825,Visa,Debit,4344676511950444,12/2022,623,YES,2,$24295,09/2002,2008,No
2731,825,Visa,Debit,4956965974959986,12/2020,393,YES,2,$21968,04/2014,2014,No
3701,825,Visa,Debit,4582313478255491,02/2024,719,YES,2,$46414,07/2003,2004,No
42,825,Visa,Credit,4879494103069057,08/2024,693,NO,1,$12400,01/2003,2012,No
4659,825,Mastercard,Debit (Prepaid),5722874738736011,03/2009,75,YES,1,$28,09/2008,2009,No
4537,1746,Visa,Credit,4404898874682993,09/2003,736,YES,1,$27500,09/2003,2012,No
1278,1746,Visa,Debit,4001482973848631,07/2022,972,YES,2,$28508,02/2011,2011,No
3687,1746,Mastercard,Debit,5627220683410948,06/2022,48,YES,2,$9022,07/2003,2015,No
3465,1746,Mastercard,Debit (Prepaid),5711382187309326,11/2020,722,YES,2,$54,06/2010,2015,No
3754,1746,Mastercard,Debit (Prepaid),5766121508358701,02/2023,908,YES,1,$99,07/2006,2012,No
